<a href="https://colab.research.google.com/github/IanTuitoekk/Shift-Handover-Management-System-NLP/blob/feature%2Fmodel-training/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shift-Handover NLP — Model Training & Evaluation

Joint mBERT fine-tuning for incident classification and named entity
recognition on multilingual (English/Swahili/code-switched) aircraft
shift-handover narratives.

This notebook restores the final selected model (v5) and runs the
official held-out test-set evaluation. Training experiments and
intermediate validation-set diagnostics are documented in the
dissertation (Chapter 5) rather than re-run here.

In [8]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ NONE'}")
print(f"torch version: {torch.__version__}")

GPU: Tesla T4
torch version: 2.11.0+cu128


In [9]:
import sys, os, shutil

!pip install -q transformers==4.44.0 scikit-learn gdown 2>&1 | tail -5

# Always remove and re-clone seqeval fresh to avoid stale/partial state
if os.path.exists('/content/seqeval_src'):
    shutil.rmtree('/content/seqeval_src')
!git clone -q https://github.com/chakki-works/seqeval.git /content/seqeval_src

sys.path.insert(0, '/content/seqeval_src')

# Clear any previously cached failed import
for mod in list(sys.modules.keys()):
    if 'seqeval' in mod:
        del sys.modules[mod]

from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score as sklearn_f1
from seqeval.metrics import f1_score as seqeval_f1, classification_report as seqeval_report

print(f"torch:        {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
print("transformers: ✓")
print("sklearn:      ✓")
print("seqeval:      ✓")

  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
torch:        2.11.0+cu128
CUDA:         True
transformers: ✓
sklearn:      ✓
seqeval:      ✓


In [10]:
REPO   = "https://github.com/IanTuitoekk/Shift-Handover-Management-System-NLP.git"
BRANCH = "feature/model-training"

if os.path.exists("/content/shift-handover-nlp"):
    %cd /content/shift-handover-nlp
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO} /content/shift-handover-nlp
    %cd /content/shift-handover-nlp

!git log --oneline -3

# Train/validation (abbreviation-expanded, used during training/tuning)
!gdown 1AIXtyfady3AxhLDVJX8VTnE_u966dPDJ -O /content/train_processed.jsonl
!gdown 173BRqvJbRhEqYT9UdB-hxs9hB9GWtDJ0 -O /content/validation_processed.jsonl

# Test (raw, unexpanded -- see process_test_corpus.py for why)
!gdown 1uGEglrA9XmFVI04yPW9D_ahXEwze8UMK -O /content/test_processed.jsonl

!ls -lh /content/*.jsonl

/content/shift-handover-nlp
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.54 KiB | 1.54 MiB/s, done.
From https://github.com/IanTuitoekk/Shift-Handover-Management-System-NLP
 * branch            feature/model-training -> FETCH_HEAD
   1c3ac5a..181cdbe  feature/model-training -> origin/feature/model-training
Updating 1c3ac5a..181cdbe
Fast-forward
 src/data_processing/process_test_corpus.py | 90 ++++++++++++++++++++++++++++++
 1 file changed, 90 insertions(+)
 create mode 100644 src/data_processing/process_test_corpus.py
181cdbe (HEAD -> feature/model-training, origin/feature/model-training) Add standalone test-set processing script (no abbreviation expansion, per eval design)
1c3ac5a Add NER token-class weighting to de-emphasize dominant O tag
78e5f7d Add differential LR for heads, increase dropout, add 

In [11]:
CHECKPOINT_DIR = "/content/model_output_v5"

if not os.path.exists(CHECKPOINT_DIR):
    !gdown 1qdHx_60mJLBmJlXjKm3znykau3qst-bI -O /content/classification_ner_final_v5.zip
    import zipfile
    with zipfile.ZipFile('/content/classification_ner_final_v5.zip', 'r') as z:
        z.extractall(CHECKPOINT_DIR)

print(os.listdir(CHECKPOINT_DIR))

['label_maps.json', 'best_model.pt', 'best_metrics.json', 'training_log.json']


## Final Test-Set Evaluation

This is the official, reported evaluation for the capstone dissertation.

**Model used:** `model_output_v5` — selected after eight training experiments
exploring mitigations for severe class imbalance in the classification task
(the "Other/Rare Ground Event" category had only 15 training examples vs.
354 and 315 for the other two categories). This configuration combines:
- A `WeightedRandomSampler` to oversample the minority class per batch
- A moderate 5× class weight on the classification loss
- Differential learning rates (10× higher for the randomly-initialized
  classification/NER heads vs. the pretrained mBERT encoder)
- Increased dropout (0.3) and weight decay (0.01) to address overfitting
  observed across all prior runs

**Data:** `test.jsonl` was held out from every stage of model development —
it was never used for training, validation-based hyperparameter selection,
or model checkpointing. It was tokenized and BIO-tagged separately via
`process_test_corpus.py`, which deliberately **skips the abbreviation-
expansion step** used for train/validation (per `process_corpus.py`'s
documented design intent), so this evaluation reflects performance on
realistic, unprocessed shift-handover text.

**Why these numbers differ from validation-set results seen during tuning:**
validation F1 for this model reached ~0.70 (classification) / ~0.64 (entity)
during experimentation. The test results below are lower, particularly for
the minority "Other/Rare Ground Event" category, whose recall reverted to
its pre-mitigation baseline (0.33). This is discussed in the dissertation
as evidence that improvements on a small (n=147) validation set do not
always generalize when the affected class has an extremely low absolute
sample count (3 test examples), rather than as a failure of the mitigation
techniques themselves.

**This cell is run once, as the final step.** No further tuning or
retraining should be evaluated against `test_processed.jsonl` — doing so
would invalidate it as a held-out measure.

In [12]:
import os
if not os.path.exists("/content/model_output_v5"):
    !gdown 1qdHx_60mJLBmJlXjKm3znykau3qst-bI -O /content/classification_ner_final_v5.zip
    import zipfile
    with zipfile.ZipFile('/content/classification_ner_final_v5.zip', 'r') as z:
        z.extractall('/content/model_output_v5')

print(os.listdir("/content/model_output_v5"))

['label_maps.json', 'best_model.pt', 'best_metrics.json', 'training_log.json']


## Final Test-Set Evaluation

This is the official, reported evaluation for the capstone dissertation.

**Model used:** `model_output_v5` — selected after eight training experiments
exploring mitigations for severe class imbalance in the classification task
(the "Other/Rare Ground Event" category had only 15 training examples vs.
354 and 315 for the other two categories). This configuration combines:
- A `WeightedRandomSampler` to oversample the minority class per batch
- A moderate 5× class weight on the classification loss
- Differential learning rates (10× higher for the randomly-initialized
  classification/NER heads vs. the pretrained mBERT encoder)
- Increased dropout (0.3) and weight decay (0.01) to address overfitting
  observed across all prior runs

**Data:** `test.jsonl` was held out from every stage of model development —
it was never used for training, validation-based hyperparameter selection,
or model checkpointing. It was tokenized and BIO-tagged separately via
`process_test_corpus.py`, which deliberately **skips the abbreviation-
expansion step** used for train/validation (per `process_corpus.py`'s
documented design intent), so this evaluation reflects performance on
realistic, unprocessed shift-handover text.

**Why these numbers differ from validation-set results seen during tuning:**
validation F1 for this model reached ~0.70 (classification) / ~0.64 (entity)
during experimentation. The test results below are lower, particularly for
the minority "Other/Rare Ground Event" category, whose recall reverted to
its pre-mitigation baseline (0.33). This is discussed in the dissertation
as evidence that improvements on a small (n=147) validation set do not
always generalize when the affected class has an extremely low absolute
sample count (3 test examples), rather than as a failure of the mitigation
techniques themselves.

**This cell is run once, as the final step.** No further tuning or
retraining should be evaluated against `test_processed.jsonl` — doing so
would invalidate it as a held-out measure.

In [13]:
import json
import torch.nn as nn
from transformers import AutoModel
from collections import defaultdict

MODEL_NAME = "google-bert/bert-base-multilingual-cased"

with open(f"{CHECKPOINT_DIR}/label_maps.json") as f:
    label_maps = json.load(f)

CATEGORY_LABELS = label_maps["category_labels"]
BIO_LABELS = label_maps["bio_labels"]
CATEGORY_TO_ID = {c: i for i, c in enumerate(CATEGORY_LABELS)}
BIO_TO_ID = {t: i for i, t in enumerate(BIO_LABELS)}
ID_TO_BIO = {i: t for t, i in BIO_TO_ID.items()}

class MultiTaskMBERT(nn.Module):
    def __init__(self, encoder, num_categories, num_bio_labels, dropout=0.3):
        super().__init__()
        self.encoder = encoder
        hidden_size = encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classification_head = nn.Linear(hidden_size, num_categories)
        self.ner_head = nn.Linear(hidden_size, num_bio_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        category_logits = self.classification_head(self.dropout(outputs.pooler_output))
        ner_logits = self.ner_head(self.dropout(outputs.last_hidden_state))
        return category_logits, ner_logits

device = torch.device("cuda")
encoder = AutoModel.from_pretrained(MODEL_NAME)
model = MultiTaskMBERT(encoder, len(CATEGORY_LABELS), len(BIO_LABELS)).to(device)
model.load_state_dict(torch.load(f"{CHECKPOINT_DIR}/best_model.pt"))
model.eval()

records = [json.loads(l) for l in open("/content/test_processed.jsonl")]

true_cat, pred_cat = [], []
true_ner, pred_ner = [], []
by_lang = defaultdict(lambda: {"true_cat": [], "pred_cat": [], "true_ner": [], "pred_ner": []})

with torch.no_grad():
    for r in records:
        cat = r.get("incident_category_grouped")
        lang = r.get("language_variant", "Unknown")
        if cat not in CATEGORY_TO_ID:
            continue

        ids = torch.tensor([r["input_ids"]]).to(device)
        mask = torch.ones_like(ids).to(device)
        cat_logits, ner_logits = model(ids, mask)

        pred = cat_logits.argmax(dim=-1).item()
        true_cat.append(CATEGORY_TO_ID[cat])
        pred_cat.append(pred)
        by_lang[lang]["true_cat"].append(CATEGORY_TO_ID[cat])
        by_lang[lang]["pred_cat"].append(pred)

        bio_true = r["bio_tags"]
        bio_pred_ids = ner_logits.argmax(dim=-1)[0].cpu().tolist()
        bio_pred = [ID_TO_BIO[i] for i in bio_pred_ids[:len(bio_true)]]
        true_ner.append(bio_true)
        pred_ner.append(bio_pred)
        by_lang[lang]["true_ner"].append(bio_true)
        by_lang[lang]["pred_ner"].append(bio_pred)

print("=" * 70)
print(f"FINAL TEST SET RESULTS (n={len(true_cat)}, never used in training or tuning)")
print("=" * 70)

print("\n--- CLASSIFICATION ---")
print(sklearn_report(true_cat, pred_cat, target_names=CATEGORY_LABELS, zero_division=0))

print("\n--- ENTITY RECOGNITION (seqeval, entity-level) ---")
print(f"Entity F1: {seqeval_f1(true_ner, pred_ner):.4f}")
print(seqeval_report(true_ner, pred_ner, zero_division=0))

print("\n" + "=" * 70)
print("PER-LANGUAGE BREAKDOWN (TEST SET)")
print("=" * 70)
for lang in sorted(by_lang.keys()):
    data = by_lang[lang]
    n = len(data["true_cat"])
    cat_f1 = sklearn_f1(data["true_cat"], data["pred_cat"], average="macro", zero_division=0)
    ent_f1 = seqeval_f1(data["true_ner"], data["pred_ner"])
    print(f"{lang:<15} n={n:<4} Category F1: {cat_f1:.4f}   Entity F1: {ent_f1:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FINAL TEST SET RESULTS (n=147, never used in training or tuning)

--- CLASSIFICATION ---
                                    precision    recall  f1-score   support

        Aircraft Equipment Problem       0.65      0.65      0.65        78
Deviation/Discrepancy - Procedural       0.57      0.59      0.58        66
           Other/Rare Ground Event       1.00      0.33      0.50         3

                          accuracy                           0.62       147
                         macro avg       0.74      0.53      0.58       147
                      weighted avg       0.62      0.62      0.62       147


--- ENTITY RECOGNITION (seqeval, entity-level) ---
Entity F1: 0.6080
                        precision    recall  f1-score   support

              AIRCRAFT       0.90      0.96      0.93       125
             COMPONENT       0.53      0.64      0.58       497
              LOCATION       0.54      0.58      0.56        92
P2-1 panel connections       0.00      0.00      

In [14]:
from google.colab import files
import shutil
shutil.make_archive('/content/classification_ner_final_v5', 'zip', CHECKPOINT_DIR)
files.download('/content/classification_ner_final_v5.zip')

KeyboardInterrupt: 